In [0]:
from pyspark.sql.functions import current_timestamp, col


BUCKET = "finstream-data-ingestion"

TABLES = [
    "customers",
    "accounts",
    "merchants",
    "exchange_rates",
    "transactions",
    "transaction_events"
]


def ingest_to_bronze(table_name):

    source_path = (
        f"s3://{BUCKET}/raw/{table_name}/"
    )

    target_table = (
        f"finstream_data_pipeline.bronze.{table_name}"
    )

    print(f"Reading: {source_path}")

    df = (
        spark.read
        .parquet(source_path)
        .withColumn(
            "_ingestion_timestamp",
            current_timestamp()
        )
        .withColumn(
            "_source_file",
            col("_metadata.file_path")
        )
    )

    print(
        f"Rows read: {df.count():,}"
    )

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(target_table)
    )

    print(
        f"Successfully created: {target_table}"
    )


for table in TABLES:
    ingest_to_bronze(table)

Reading: s3://finstream-data-ingestion/raw/customers/
Rows read: 10,000
Successfully created: finstream_data_pipeline.bronze.customers
Reading: s3://finstream-data-ingestion/raw/accounts/
Rows read: 15,000
Successfully created: finstream_data_pipeline.bronze.accounts
Reading: s3://finstream-data-ingestion/raw/merchants/
Rows read: 2,000
Successfully created: finstream_data_pipeline.bronze.merchants
Reading: s3://finstream-data-ingestion/raw/exchange_rates/
Rows read: 10,950
Successfully created: finstream_data_pipeline.bronze.exchange_rates
Reading: s3://finstream-data-ingestion/raw/transactions/
Rows read: 100,000
Successfully created: finstream_data_pipeline.bronze.transactions
Reading: s3://finstream-data-ingestion/raw/transaction_events/
Rows read: 291,967
Successfully created: finstream_data_pipeline.bronze.transaction_events
